# What the allocation code should do

At a minimum, the allocation engine should have 5 stages:

1. **Read request**
2. **Filter feasible spaces**
3. **Score candidates**
4. **Pick best candidate**
5. **Commit atomically**



## Core shape of the code

In [1]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Optional, Dict, Tuple


@dataclass
class BookingRequest:
    user_id: str
    site_id: str
    start: datetime
    end: datetime
    vehicle_type: str              # "ev", "ice", "van", ...
    connector_needed: Optional[str] = None
    min_power_kw: Optional[float] = None
    requires_accessibility: bool = False
    user_priority: int = 0         # internal staff > guest > public
    purpose: str = "general"       # "employee", "visitor", "delivery", ...


@dataclass
class Space:
    id: str
    site_id: str
    zone_id: str
    is_active: bool
    vehicle_types_allowed: List[str]
    ev_only: bool
    connectors: List[str] = field(default_factory=list)
    max_power_kw: float = 0.0
    accessible: bool = False
    reservable: bool = True
    shared_public: bool = False
    walking_distance_m: float = 0.0
    rules: Dict = field(default_factory=dict)


@dataclass
class Reservation:
    space_id: str
    start: datetime
    end: datetime
    status: str  # "confirmed", "held", "cancelled"


@dataclass
class ForecastSignal:
    expected_occupancy_ratio: float     # 0.0 to 1.0
    expected_internal_demand: int
    confidence: float                   # 0.0 to 1.0


@dataclass
class AllocationDecision:
    accepted: bool
    space_id: Optional[str]
    reason: str
    score_breakdown: Dict[str, float] = field(default_factory=dict)

## Feasibility first

This is where hard business rules are enforcing.


In [ ]:
def overlaps(a_start, a_end, b_start, b_end) -> bool:
    return a_start < b_end and b_start < a_end


def is_available(space: Space, reservations: List[Reservation], req: BookingRequest) -> bool:
    for r in reservations:
        if r.space_id == space.id and r.status in ("confirmed", "held"):
            if overlaps(r.start, r.end, req.start, req.end):
                return False
    return True


def passes_hard_constraints(space: Space, req: BookingRequest) -> Tuple[bool, str]:
    if not space.is_active:
        return False, "space_inactive"

    if not space.reservable:
        return False, "not_reservable"

    if req.vehicle_type not in space.vehicle_types_allowed:
        return False, "vehicle_type_not_allowed"

    if req.requires_accessibility and not space.accessible:
        return False, "accessibility_required"

    if req.vehicle_type != "ev" and space.ev_only:
        return False, "ev_only_space"

    if req.vehicle_type == "ev":
        if req.connector_needed and req.connector_needed not in space.connectors:
            return False, "connector_missing"
        if req.min_power_kw and space.max_power_kw < req.min_power_kw:
            return False, "insufficient_power"

    return True, "ok"

## Score Candidates

Scoring the feasible candidates, is where forecasting enters. A forecast should influence allocation, not replace it.

Example scoring logic:

* penalize assigning scarce EV fast chargers to low-need users
* penalize using reserve capacity if forecast says internal demand spike is coming
* reward closer spaces for short visits
* reward policy fit, like guest spaces for guests
* reward spaces in zones currently meant to be opened


In [ ]:
def score_space(
    space: Space,
    req: BookingRequest,
    forecast: ForecastSignal,
    zone_current_load: float,
    zone_capacity: int
) -> Dict[str, float]:
    scores = {}

    # 1. Fit to request
    scores["fit"] = 50.0

    if req.vehicle_type == "ev":
        if req.min_power_kw:
            scores["power_fit"] = min(space.max_power_kw / max(req.min_power_kw, 1), 2.0) * 10
        else:
            scores["power_fit"] = 5.0
    else:
        scores["power_fit"] = 0.0

    # 2. Distance preference
    scores["distance_penalty"] = -0.03 * space.walking_distance_m

    # 3. Preserve scarce premium spaces
    if space.max_power_kw >= 100 and req.min_power_kw and req.min_power_kw < 50:
        scores["scarcity_penalty"] = -15.0
    else:
        scores["scarcity_penalty"] = 0.0

    # 4. Forecast-aware reserve logic
    # if expected occupancy is high, be more conservative with shared/public spaces
    if forecast.expected_occupancy_ratio > 0.85:
        scores["reserve_penalty"] = -20.0 if space.shared_public else 0.0
    else:
        scores["reserve_penalty"] = 0.0

    # 5. Keep zones balanced
    utilization_ratio = zone_current_load / max(zone_capacity, 1)
    scores["load_balance"] = -(utilization_ratio * 10.0)

    # 6. Policy preference
    if req.purpose == "visitor" and space.rules.get("preferred_for_visitors"):
        scores["policy_bonus"] = 8.0
    elif req.purpose == "employee" and space.rules.get("preferred_for_employees"):
        scores["policy_bonus"] = 8.0
    else:
        scores["policy_bonus"] = 0.0

    # 7. Confidence-aware caution
    scores["forecast_confidence_adjustment"] = -5.0 if forecast.confidence < 0.5 else 0.0

    return scores


def total_score(score_breakdown: Dict[str, float]) -> float:
    return sum(score_breakdown.values())

## Full allocator

In [ ]:
def allocate_space(
    req: BookingRequest,
    spaces: List[Space],
    reservations: List[Reservation],
    forecasts_by_zone: Dict[str, ForecastSignal],
    zone_loads: Dict[str, int],
    zone_capacities: Dict[str, int]
) -> AllocationDecision:

    candidates = []

    for space in spaces:
        if space.site_id != req.site_id:
            continue

        ok, reason = passes_hard_constraints(space, req)
        if not ok:
            continue

        if not is_available(space, reservations, req):
            continue

        forecast = forecasts_by_zone.get(
            space.zone_id,
            ForecastSignal(expected_occupancy_ratio=0.5, expected_internal_demand=0, confidence=0.0)
        )

        breakdown = score_space(
            space=space,
            req=req,
            forecast=forecast,
            zone_current_load=zone_loads.get(space.zone_id, 0),
            zone_capacity=zone_capacities.get(space.zone_id, 1),
        )

        candidates.append((space, breakdown, total_score(breakdown)))

    if not candidates:
        return AllocationDecision(
            accepted=False,
            space_id=None,
            reason="no_feasible_space"
        )

    candidates.sort(key=lambda x: x[2], reverse=True)
    best_space, breakdown, _ = candidates[0]

    return AllocationDecision(
        accepted=True,
        space_id=best_space.id,
        reason="allocated",
        score_breakdown=breakdown
    )

## Reservation Commit

If two users ask for the same slot at the same moment, you need a lock or transactional write. Otherwise the allocator becomes a double-booking vending machine.

In [ ]:
def commit_reservation(decision: AllocationDecision, req: BookingRequest, db) -> bool:
    if not decision.accepted or not decision.space_id:
        return False

    with db.transaction():
        # re-check availability inside the transaction
        still_free = db.space_is_available(decision.space_id, req.start, req.end)
        if not still_free:
            return False

        db.insert_reservation(
            space_id=decision.space_id,
            user_id=req.user_id,
            start=req.start,
            end=req.end,
            status="confirmed"
        )
        return True

## Where the forecast plugs in

Using the logic from the posts, the prediction service can produce, for each site or zone:

* expected occupancy in 1 hour
* expected occupancy in 3 hours
* expected internal demand
* confidence band

The allocator should consume that as policy input, for example:

* If predicted occupancy > 90%, do not expose overflow spaces publicly
* If predicted internal demand spike starts at 08:30, stop external bookings at 08:00
* If evening underuse is highly likely, open a temporary public booking window
* If uncertainty is high, keep a reserve buffer

So the real bridge is:

In [ ]:
if forecast.expected_occupancy_ratio < 0.60 and forecast.confidence > 0.75:
    open_public_inventory(zone_id)

if forecast.expected_occupancy_ratio > 0.90:
    restrict_low_priority_bookings(zone_id)

## Three Layers for AlgoTecture

Building **three layers**.

### 1. Rule engine

Pure deterministic logic.

* EV-only or mixed-use
* internal vs external priority
* bookable hours
* access rights
* connector / power / vehicle compatibility
* max dwell time
* reserve thresholds

### 2. Scoring engine

Weighted ranking among feasible spaces.

* walking distance
* preserve premium chargers
* zone balancing
* expected future pressure
* user priority
* monetization logic

### 3. Optional optimizer

Only later, for batch allocation across many simultaneous requests.

At that stage, use:

* Hungarian algorithm for one-shot assignment
* Mixed Integer Programming if you want hard optimization across zones, times, priorities, and capacities

For an MVP, **greedy ranking + transaction lock** is enough.

## Minimal AlgoTecture architecture

The codebase should probably be split like this:

* `forecast_service`

  * trains and serves occupancy predictions
* `inventory_service`

  * spaces, zones, attributes, status
* `rules_engine`

  * hard business rules
* `allocation_engine`

  * filtering + scoring + selection
* `reservation_service`

  * holds, confirmations, cancellations
* `event_log`

  * every decision stored for learning and audit

That last one matters. Every allocation decision should generate an event like:

```json
{
  "request_id": "r_123",
  "site_id": "site_a",
  "space_id": "space_42",
  "decision": "allocated",
  "timestamp": "2026-04-01T16:00:00Z",
  "features": {
    "predicted_occupancy": 0.72,
    "vehicle_type": "ev",
    "priority": 2
  },
  "score_breakdown": {
    "fit": 50,
    "distance_penalty": -3.6,
    "reserve_penalty": 0,
    "policy_bonus": 8
  }
}
```

That gives you explainability, tuning material, and future training data.

## The blunt version

The allocation, as the actual AlgoTecture's core product, is a **decision engine**, starting one layer above estimating a future occupancy from historical and contextual variables. frames multivariate time series for LSTM learning, and EBP uses that pattern for short-horizon parking forecasts with current occupancy plus contextual features; within the following structure:

* filter what is allowed
* rank what is feasible
* reserve what is best
* log what happened
* learn from the result

